In [4]:
import os
from moviepy.video.io.VideoFileClip import VideoFileClip

def compress_video_moviepy(input_path, output_path, target_size_mb, bitrate=None):
    """
    Compress video using moviepy
    Args:
        input_path: Path to input video
        output_path: Path for output video
        target_size_mb: Target size in MB
        bitrate: Optional bitrate in kbps (auto-calculated if None)
    """
    try:
        # Load video
        clip = VideoFileClip(input_path)
        
        # Get video duration in seconds
        duration = clip.duration
        
        # Calculate target bitrate if not provided
        if bitrate is None:
            # Formula: bitrate (kbps) = (target_size_mb * 8192) / duration
            target_size_kb = target_size_mb * 1024
            bitrate = int((target_size_kb * 8) / duration)
            # Ensure minimum bitrate
            bitrate = max(bitrate, 500)  # Minimum 500 kbps
        
        print(f"Original video: {os.path.getsize(input_path) / (1024*1024):.2f} MB")
        print(f"Target size: {target_size_mb} MB")
        print(f"Calculated bitrate: {bitrate} kbps")
        
        # Write compressed video
        clip.write_videofile(
            output_path,
            bitrate=f"{bitrate}k",
            audio_bitrate="128k",
            codec="libx264",
            audio_codec="aac"
        )
        
        original_size = os.path.getsize(input_path) / (1024*1024)
        compressed_size = os.path.getsize(output_path) / (1024*1024)
        
        print(f"\nCompression complete!")
        print(f"Original: {original_size:.2f} MB")
        print(f"Compressed: {compressed_size:.2f} MB")
        print(f"Reduction: {(1 - compressed_size/original_size)*100:.1f}%")
        
        clip.close()
        
    except Exception as e:
        print(f"Error: {e}")

# Alternative method with more control
def compress_video_ffmpeg_python(input_path, output_path, scale_factor=0.5, crf=23):
    """
    Alternative method using ffmpeg-python library
    Args:
        input_path: Input video path
        output_path: Output video path
        scale_factor: Scale resolution (0.5 = half size)
        crf: Constant Rate Factor (18-28 for H.264, lower = better quality)
    """
    import ffmpeg
    
    try:
        # Get video info
        probe = ffmpeg.probe(input_path)
        video_info = next(s for s in probe['streams'] if s['codec_type'] == 'video')
        
        original_width = int(video_info['width'])
        original_height = int(video_info['height'])
        
        # Calculate new dimensions
        new_width = int(original_width * scale_factor)
        new_height = int(original_height * scale_factor)
        
        # Ensure even dimensions (required by some codecs)
        new_width = new_width - (new_width % 2)
        new_height = new_height - (new_height % 2)
        
        print(f"Original resolution: {original_width}x{original_height}")
        print(f"New resolution: {new_width}x{new_height}")
        
        # Compress using ffmpeg
        (
            ffmpeg
            .input(input_path)
            .output(output_path, 
                   vf=f'scale={new_width}:{new_height}',
                   vcodec='libx264',
                   crf=crf,
                   acodec='aac',
                   audio_bitrate='128k')
            .run(quiet=True, overwrite_output=True)
        )
        
        original_size = os.path.getsize(input_path) / (1024*1024)
        compressed_size = os.path.getsize(output_path) / (1024*1024)
        
        print(f"\nCompression complete!")
        print(f"Original: {original_size:.2f} MB")
        print(f"Compressed: {compressed_size:.2f} MB")
        print(f"Reduction: {(1 - compressed_size/original_size)*100:.1f}%")
        
    except Exception as e:
        print(f"Error: {e}")
        
compress_video_moviepy(
    input_path="video.mp4",
    output_path="output.mp4",
    target_size_mb=10
)

Original video: 667.76 MB
Target size: 10 MB
Calculated bitrate: 500 kbps
MoviePy - Building video output.mp4.
MoviePy - Writing audio in outputTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
MoviePy - Writing video output.mp4



MoviePy - Done !
MoviePy - video ready output.mp4

Compression complete!
Original: 667.76 MB
Compressed: 45.97 MB
Reduction: 93.1%
